# Biblioteca de Caminhadas Quânticas em Grafos Toroidais

Este notebook demonstra o uso de uma biblioteca Python para simulação de caminhadas quânticas em grafos toroidais, enfatizando o armazenamento dos resultados em JSON Server (`db.json`).

## Seções
1. Configuração do Ambiente e Instalação de Dependências
2. Estrutura do Projeto e Armazenamento em JSON Server
3. Implementação do Módulo `graph.py`
4. Implementação do Módulo `walker.py`
5. Implementação do Módulo `simulation.py`
6. Configuração e Execução de Simulações
7. Armazenamento e Recuperação de Resultados no JSON Server
8. Visualização Gráfica e Animação dos Resultados
9. Testes Unitários com pytest

## 1. Configuração do Ambiente e Instalação de Dependências

Instale as dependências necessárias para rodar a biblioteca e o notebook. Certifique-se de que o JSON Server está instalado e rodando para o armazenamento dos resultados em `db.json`.

In [ ]:
# Instalação das dependências (execute apenas uma vez)
# !pip install numpy matplotlib requests pytest
# Instale o JSON Server (Node.js):
# !npm install -g json-server
# Inicie o JSON Server em outro terminal:
# !json-server --watch db.json

import numpy as np
import matplotlib.pyplot as plt
import requests

: 

## 2. Estrutura do Projeto e Armazenamento em JSON Server

A biblioteca está organizada em três módulos principais:
- `graph.py`: estrutura da grade toroidal e vizinhos
- `walker.py`: estado e evolução do caminhante quântico
- `simulation.py`: execução da simulação e integração com o JSON Server

Os resultados das simulações são armazenados no arquivo `db.json` via requisições HTTP para o JSON Server, permitindo fácil persistência e recuperação dos dados.

## 3. Implementação do Módulo graph.py

O módulo `graph.py` define a estrutura da grade toroidal, permitindo configurar o número de dimensões, tamanho da grade, self-loops e pesos das arestas. Também fornece métodos para obter os vizinhos de cada vértice.

In [ ]:
# Exemplo de uso do módulo graph.py
from biblioteca.graph import ToroidalGrid

grid = ToroidalGrid(L=3, n=2, num_selfloop=1, weight_value=1.0)
print('Matriz de adjacência:')
print(grid.adj_matrix)
print('Vizinhos do vértice 0:', grid.get_neighbors(0))

## 4. Implementação do Módulo walker.py

O módulo `walker.py` implementa o estado do caminhante quântico e sua evolução temporal, permitindo calcular as probabilidades de detecção nos vértices marcados.

In [ ]:
# Exemplo de uso do módulo walker.py
from biblioteca.walker import QuantumWalker

grid = ToroidalGrid(L=3, n=2)
walker = QuantumWalker(grid, marked_vertices=[0, 1])
print('Estado inicial:', walker.state)
walker.step()
print('Estado após 1 passo:', walker.state)
print('Probabilidades:', walker.measure_probabilities())
print('Probabilidade de detecção:', walker.detection_probability())

## 5. Implementação do Módulo simulation.py

O módulo `simulation.py` integra os módulos anteriores, executa a simulação parametrizada e armazena os resultados no JSON Server (`db.json`).

In [ ]:
# Exemplo de uso do módulo simulation.py
from biblioteca.simulation import QuantumWalkSimulation

sim = QuantumWalkSimulation(L=3, n=2, num_selfloop=1, t_f=10, weight_value=1.0, marked_vertices=[0, 1])
probs, det_times = sim.run()
print('Probabilidades shape:', probs.shape)
print('Detecção shape:', det_times.shape)

## 6. Configuração e Execução de Simulações

A seguir, configuramos diferentes parâmetros e executamos simulações, analisando os resultados retornados em arrays NumPy.

In [ ]:
# Configurando e executando simulações com diferentes parâmetros
params = [
    {'L': 3, 'n': 2, 'num_selfloop': 1, 't_f': 10, 'weight_value': 1.0, 'marked_vertices': [0]},
    {'L': 3, 'n': 2, 'num_selfloop': 2, 't_f': 10, 'weight_value': 1.0, 'marked_vertices': [1]},
]
results = []
for p in params:
    sim = QuantumWalkSimulation(**p)
    probs, det_times = sim.run()
    results.append({'params': p, 'probs': probs, 'det_times': det_times})
    print(f"Simulação com {p}: Última prob. de detecção = {det_times[-1]:.4f}")

## 7. Armazenamento e Recuperação de Resultados no JSON Server

Os resultados das simulações são enviados automaticamente ao JSON Server (`db.json`). Para recuperar os dados, basta fazer uma requisição HTTP GET ao endpoint do servidor.

In [ ]:
# Recuperando resultados do JSON Server
url = 'http://localhost:3000/results'
try:
    response = requests.get(url)
    if response.ok:
        data = response.json()
        print(f"Total de simulações armazenadas: {len(data)}")
        print("Exemplo de resultado:", data[-1] if data else None)
    else:
        print('Erro ao acessar o JSON Server:', response.status_code)
except Exception as e:
    print('JSON Server não está rodando:', e)

## 8. Visualização Gráfica e Animação dos Resultados

Utilize o matplotlib para visualizar a evolução das probabilidades e dos tempos de detecção ao longo dos passos da simulação.

In [ ]:
# Visualização dos resultados da última simulação
import matplotlib.animation as animation

last = results[-1]
probs = last['probs']
det_times = last['det_times']

plt.figure(figsize=(10,4))
plt.subplot(1,2,1)
plt.plot(det_times)
plt.title('Probabilidade de Detecção')
plt.xlabel('Passo')
plt.ylabel('Probabilidade')

plt.subplot(1,2,2)
plt.imshow(probs.T, aspect='auto', cmap='viridis')
plt.title('Evolução das Probabilidades')
plt.xlabel('Passo')
plt.ylabel('Vértice')
plt.colorbar(label='Probabilidade')
plt.tight_layout()
plt.show()

# Animação da evolução do estado
fig, ax = plt.subplots()
line, = ax.plot(probs[0])
ax.set_ylim(0, 1)
ax.set_title('Evolução do Estado do Caminhante')
ax.set_xlabel('Vértice')
ax.set_ylabel('Probabilidade')

def update(frame):
    line.set_ydata(probs[frame])
    ax.set_title(f'Evolução do Estado - Passo {frame}')
    return line,

ani = animation.FuncAnimation(fig, update, frames=range(probs.shape[0]), blit=True)
plt.show()

## 9. Testes Unitários com pytest

A biblioteca inclui testes unitários para validar as principais funções dos módulos. Execute os testes com o comando abaixo no terminal.

In [ ]:
# Execute no terminal:
# !pytest tests/

# Exemplo de teste direto no notebook
import pytest
from biblioteca.graph import ToroidalGrid
from biblioteca.walker import QuantumWalker
from biblioteca.simulation import QuantumWalkSimulation

def test_graph():
    grid = ToroidalGrid(L=2, n=1)
    assert set(grid.get_neighbors(0)) == {1}

def test_walker():
    grid = ToroidalGrid(L=2, n=1)
    walker = QuantumWalker(grid)
    walker.step()
    assert np.isclose(np.sum(np.abs(walker.state)**2), 1.0)

def test_simulation():
    sim = QuantumWalkSimulation(L=2, n=1, num_selfloop=0, t_f=3, weight_value=1.0, marked_vertices=[1], db_url=None)
    probs, det_times = sim.run()
    assert probs.shape[0] == 3
    assert det_times.shape[0] == 3

# pytest.main(["-v"])
print('Testes prontos para execução com pytest.')